<a href="https://colab.research.google.com/github/pedroedu02/Challenge3_PosTech/blob/main/exploracao_CSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Carregamento dos arquivos CSV

In [3]:
import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

FILES = {
    '2023': '/content/state_of_data_2023.csv',
    '2024': '/content/state_of_data_2024.csv',
    '2025-2026': '/content/state_of_data_2025-2026.csv',
}

dfs = {}
for ano, arquivo in FILES.items():
    df = pd.read_csv(arquivo, sep=None, engine='python')
    dfs[ano] = df
    print(f"{ano}: {df.shape[0]} linhas x {df.shape[1]} colunas")


2023: 5293 linhas x 399 colunas
2024: 5217 linhas x 403 colunas
2025-2026: 3495 linhas x 388 colunas


## 2. Compara as tabelas as linhas, colunas, nulos e duplicidades

In [4]:
resumo = []
for ano, df in dfs.items():
    total_celulas = df.shape[0] * df.shape[1]
    nulos = df.isna().sum().sum()
    resumo.append({
        'ano': ano,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'nulos_totais': nulos,
        'pct_nulos': round(nulos / total_celulas * 100, 1),
        'linhas_duplicadas': df.duplicated().sum(),
    })

df_resumo = pd.DataFrame(resumo).set_index('ano')
df_resumo

,linhas,colunas,nulos_totais,pct_nulos,linhas_duplicadas
ano,,,,,
2023,5293,399,1197052,56.7,0
2024,5217,403,1202637,57.2,2
2025-2026,3495,388,839991,61.9,1


**OBSERVAÇÃO:** o alto percentual de nulos (56-62%) não é errado é da estrutura.
A pesquisa tem lógica condicional, entao quem não é gestor não responde bloco de gestão, e dezenas de perguntas de múltipla escolha viram uma coluna binária por opção.

## 3. Encontrando padrão de colunas, para achar onde podemos juntar

In [5]:
for ano, df in dfs.items():
    print(f"--- {ano}: primeiras 5 colunas ---")
    for c in df.columns[:5]:
        print(" ", c)
    print()

--- 2023: primeiras 5 colunas ---
  ('P0', 'id')
  ('P1_a ', 'Idade')
  ('P1_a_1 ', 'Faixa idade')
  ('P1_b ', 'Genero')
  ('P1_c ', 'Cor/raca/etnia')

--- 2024: primeiras 5 colunas ---
  0.a_token
  0.d_data/hora_envio
  1.a_idade
  1.a.1_faixa_idade
  1.b_genero

--- 2025-2026: primeiras 5 colunas ---
  0.a_token
  0.d_data/hora_envio
  1.a_idade
  1.a.1_faixa_idade
  1.b_genero



**OBSERVAÇÃO:** 2023 usa um código `P<bloco>_<letra>` com o nome da pergunta embutido em formato
de tupla Python (ex.: `('P1_a ', 'Idade')`). 2024 e 2025-2026 usam `<bloco>.<letra>_<slug>`
(ex.: `1.a_idade`). É obrigatório um de-para manual antes de qualquer união entre os anos.

## 4. DE PARA COLunas

In [6]:
# Cada variável de negócio -> chave de busca do nome real da coluna em cada ano
DICIONARIO = {
    'idade':                {'2023': "('P1_a '",  '2024': '1.a_idade',                        '2025-2026': '1.a_idade'},
    'genero':               {'2023': "('P1_b '",  '2024': '1.b_genero',                       '2025-2026': '1.b_genero'},
    'regiao':                {'2023': "('P1_i_2 '",'2024': '1.i.2_regiao_onde_mora',           '2025-2026': '1.i.2_regiao_onde_mora'},
    'nivel_ensino':          {'2023': "('P1_l '",  '2024': '1.l_nivel_de_ensino',              '2025-2026': '1.l_nivel_de_ensino'},
    'cargo_atual':           {'2023': "('P2_f '",  '2024': '2.f_cargo_atual',                  '2025-2026': '2.f_cargo_atual'},
    'senioridade':           {'2023': "('P2_g '",  '2024': '2.g_nivel',                        '2025-2026': '2.g_nivel'},
    'faixa_salarial':        {'2023': "('P2_h '",  '2024': '2.h_faixa_salarial',               '2025-2026': '2.h_faixa_salarial'},
    'exp_dados':             {'2023': "('P2_i '",  '2024': '2.i_tempo_de_experiencia_em_dados','2025-2026': '2.i_tempo_de_experiencia_em_dados'},
    'modelo_trabalho':       {'2023': "('P2_r '",  '2024': '2.r_modelo_de_trabalho_atual',     '2025-2026': '2.q_modelo_de_trabalho_atual'},
    'linguagem_preferida':   {'2023': "('P4_f '",  '2024': '4.f_linguagem_preferida',          '2025-2026': '4.c_linguagem_preferida'},
    'cloud_dia_a_dia':       {'2023': "('P4_h '",  '2024': '4.h_cloud_(dia_a_dia)',            '2025-2026': '4.e_cloud_(dia_a_dia)'},
    'ferramenta_bi_dia_a_dia':{'2023': "('P4_j '", '2024': '4.j_ferramenta_de_bi_(dia_a_dia)', '2025-2026': '4.g_ferramenta_de_bi_(dia_a_dia)'},
    'uso_chatgpt_copilot':   {'2023': "('P4_m '",  '2024': '4.m_usa_chatgpt_ou_copilot_no_trabalho?','2025-2026': '4.j_usa_chatgpt_ou_copilot_no_trabalho?'},
}

def find_col(columns, key):
    for c in columns:
        if c == key or c.startswith(key):
            return c
    return None

linhas_dicionario = []
for var, anos in DICIONARIO.items():
    linha = {'variavel': var}
    for ano, key in anos.items():
        col_real = find_col(dfs[ano].columns, key)
        linha[ano] = col_real
    linhas_dicionario.append(linha)

df_dicionario = pd.DataFrame(linhas_dicionario).set_index('variavel')
df_dicionario



,2023,2024,2025-2026
variavel,,,
idade,"('P1_a ', 'Idade')",1.a_idade,1.a_idade
genero,"('P1_b ', 'Genero')",1.b_genero,1.b_genero
regiao,"('P1_i_2 ', 'Regiao onde mora')",1.i.2_regiao_onde_mora,1.i.2_regiao_onde_mora
nivel_ensino,"('P1_l ', 'Nivel de Ensino')",1.l_nivel_de_ensino,1.l_nivel_de_ensino
cargo_atual,"('P2_f ', 'Cargo Atual')",2.f_cargo_atual,2.f_cargo_atual
senioridade,"('P2_g ', 'Nivel')",2.g_nivel,2.g_nivel
faixa_salarial,"('P2_h ', 'Faixa salarial')",2.h_faixa_salarial,2.h_faixa_salarial
exp_dados,"('P2_i ', 'Quanto tempo de experiência na área...",2.i_tempo_de_experiencia_em_dados,2.i_tempo_de_experiencia_em_dados
modelo_trabalho,"('P2_r ', 'Atualmente qual a sua forma de trab...",2.r_modelo_de_trabalho_atual,2.q_modelo_de_trabalho_atual


In [7]:
for var, anos in DICIONARIO.items():
    print(f"=== {var} ===")
    for ano, key in anos.items():
        col_real = find_col(dfs[ano].columns, key)
        serie = dfs[ano][col_real]
        exemplos = list(serie.dropna().unique()[:4])
        print(f"  {ano}: não-nulos={serie.notna().sum():>5} | exemplos={exemplos}")
    print()


=== idade ===
  2023: não-nulos= 5293 | exemplos=[np.int64(31), np.int64(30), np.int64(37), np.int64(22)]
  2024: não-nulos= 5217 | exemplos=[np.int64(18), np.int64(19), np.int64(20), np.int64(21)]
  2025-2026: não-nulos= 3495 | exemplos=[np.int64(35), np.int64(20), np.int64(29), np.int64(18)]

=== genero ===
  2023: não-nulos= 5293 | exemplos=['Masculino', 'Feminino', 'Outro', 'Prefiro não informar']
  2024: não-nulos= 5217 | exemplos=['Masculino', 'Feminino', 'Outro', 'Prefiro não informar']
  2025-2026: não-nulos= 3495 | exemplos=['Masculino', 'Feminino', 'Outro', 'Prefiro não informar']

=== regiao ===
  2023: não-nulos= 5169 | exemplos=['Sudeste', 'Nordeste', 'Sul', 'Centro-oeste']
  2024: não-nulos= 5075 | exemplos=['Sul', 'Sudeste', 'Centro-oeste', 'Nordeste']
  2025-2026: não-nulos= 3370 | exemplos=['Sudeste', 'Sul', 'Nordeste', 'Centro-oeste']

=== nivel_ensino ===
  2023: não-nulos= 5293 | exemplos=['Doutorado ou Phd', 'Graduação/Bacharelado', 'Estudante de Graduação', 'Pós-g

In [8]:
print("--- Categorias de SENIORIDADE por ano ---")
for ano, key in DICIONARIO['senioridade'].items():
    col_real = find_col(dfs[ano].columns, key)
    print(f"{ano}: {sorted(dfs[ano][col_real].dropna().unique().tolist())}")


--- Categorias de SENIORIDADE por ano ---
2023: ['Júnior', 'Pleno', 'Sênior']
2024: ['Júnior', 'Pleno', 'Sênior']
2025-2026: ['Especialista/Staff+', 'Júnior', 'Pleno', 'Sênior']


**OBSERVAÇÃO:** a categoria **"Especialista/Staff+"** só existe em 2025-2026. Precisa ser
tratada (ex.: agrupada com "Sênior") para permitir comparação justa nos 3 anos — decisão
já aplicada na camada Silver.

In [9]:
print("--- Bloco de tecnologia (nivel superior) 2024 vs 2025-2026 ---")
bloco4_2024 = [c for c in dfs['2024'].columns if c.startswith('4.') and c.count('.') == 1]
bloco4_2025 = [c for c in dfs['2025-2026'].columns if c.startswith('4.') and c.count('.') == 1]
print("2024:", bloco4_2024)
print()
print("2025-2026:", bloco4_2025)


--- Bloco de tecnologia (nivel superior) 2024 vs 2025-2026 ---
2024: ['4.a_funcao_de_atuacao', '4.b_fontes_de_dados_(dia_a_dia)', '4.c_fonte_de_dado_mais_usada', '4.d_linguagem_de_programacao_(dia_a_dia)', '4.e_linguagem_mais_usada', '4.f_linguagem_preferida', '4.g_banco_de_dados_(dia_a_dia)', '4.h_cloud_(dia_a_dia)', '4.i_cloud_preferida', '4.j_ferramenta_de_bi_(dia_a_dia)', '4.k_ferramenta_de_bi_preferida', '4.l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa', '4.m_usa_chatgpt_ou_copilot_no_trabalho?']

2025-2026: ['4.a_funcao_de_atuacao', '4.b_fontes_de_dados_(dia_a_dia)', '4.c_linguagem_preferida', '4.d_banco_de_dados_(dia_a_dia)', '4.e_cloud_(dia_a_dia)', '4.f_cloud_preferida', '4.g_ferramenta_de_bi_(dia_a_dia)', '4.h_ferramenta_de_bi_preferida', '4.i_tipo_de_uso_de_ai_generativa_e_llm_na_empresa', '4.j_usa_chatgpt_ou_copilot_no_trabalho?']


**OBSERVAÇÔA:** em 2024 existiam perguntas separadas para "linguagem usada no dia a dia",
"linguagem mais usada" e "linguagem preferida". Em 2025-2026 essas 3 perguntas viraram
**uma única** pergunta de múltipla escolha (`linguagem_preferida`). Isso quebra a
comparabilidade direta de "linguagem mais usada" entre 2024 e 2025-2026.

In [ ]:
print("--- Checagem de inconsistência pontual: faixa salarial 2025-2026 ---")
col_salario = find_col(dfs['2025-2026'].columns, DICIONARIO['faixa_salarial']['2025-2026'])
contagem = dfs['2025-2026'][col_salario].value_counts()
print(contagem[contagem.index.str.contains('3000', na=False)])


**OBSERVAÇÔA:** 1 registro isolado com o valor `"de R$ 25.001/mês a R$ 3000/mês"`, que não
segue o padrão das demais faixas (provável erro de digitação — deveria ser R$ 30.000).
Tratado na camada Silver.

## 7. Resumo

In [10]:
print(f"Total de respondentes (3 anos, antes de deduplicação): {sum(len(df) for df in dfs.values())}")
print()
print("Variáveis do dicionário totalmente comparáveis nos 3 anos:")
for var in DICIONARIO:
    if var not in ('linguagem_preferida',):
        print(f"  - {var}")
print()
print("Variáveis com limitação de comparabilidade:")
print("  - senioridade (categoria nova 'Especialista/Staff+' só em 2025-2026)")
print("  - linguagem_preferida (pergunta de resposta única virou múltipla escolha em 2025-2026)")

Total de respondentes (3 anos, antes de deduplicação): 14005

Variáveis do dicionário totalmente comparáveis nos 3 anos:
  - idade
  - genero
  - regiao
  - nivel_ensino
  - cargo_atual
  - senioridade
  - faixa_salarial
  - exp_dados
  - modelo_trabalho
  - cloud_dia_a_dia
  - ferramenta_bi_dia_a_dia
  - uso_chatgpt_copilot

Variáveis com limitação de comparabilidade:
  - senioridade (categoria nova 'Especialista/Staff+' só em 2025-2026)
  - linguagem_preferida (pergunta de resposta única virou múltipla escolha em 2025-2026)
